In [1]:
import os, json, requests
import pandas as pd
import numpy as np
from urllib.parse import quote
import time, random

import re
from tqdm.auto import tqdm
import ast

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from requests.exceptions import HTTPError, RequestException

from dotenv import load_dotenv

/data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()  # 기본: 현재 폴더의 .env 로드

S2_API_KEY = os.getenv("S2_API_KEY")

## API

### API를 통해 AI 관련 논문들을 뽑아옴 (일단 1000개)
- `id`, `title`, `fieldsOfStudy`, `s2FieldsOfStudy`, `year`, `url`, `abstract`, `publication (= venue)`, `referenceCount`, `citationCount`
- https://api.semanticscholar.org/api-docs/#tag/Paper-Data/operation/get_graph_paper_relevance_search

In [ ]:
!curl -G "https://api.semanticscholar.org/graph/v1/paper/search/bulk" \
  --data-urlencode 'query=AI | "machine learning" | "deep learning" | "large language model" | NLP | "natural language processing" | CV | "computer vision" | "multi-modal"' \
  --data-urlencode 'limit=1' \
  --data-urlencode 'sort=citationCount:desc' \
  --data-urlencode 'fields=paperId,title,year,url,abstract,venue,referenceCount,citationCount,fieldsOfStudy,s2FieldsOfStudy' \
  -H "x-api-key: $S2_API_KEY" \
  -o papers.json

## 만약 이어서 받고 싶다면 --data-urlencode 'token={이전 응답의 token 값}'  추가

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2061k  100 2061k    0     0   265k      0  0:00:07  0:00:07 --:--:--  498k


In [ ]:
with open("papers.json", "r", encoding="utf-8") as f:
    res = json.load(f)

print("total:", res.get("total"))  ## query 조건에 매칭되는 전체 논문 수(추정치)
print("data_len:", len(res.get("data", [])))  ## 실제로 내려받은 논문 수
print("token:", res.get("token"))

## 기본적으로 1000개 이상 limit 하라고 했기 때문에 1로 했어도 1000개

total: 2495941
data_len: 1000
token: PCOA3RZBB2ADADAF2CVSZVMISWZPLQ2VBCRGZHJBIEAI44DXE2PHQL7NKYHD6KZNQH6KMIMQWXD27XUVYHNN7JW2CQGBSMZHQEVSHZRMYKMODHUKKXWWFIXN7MAUTNIVAQ


In [6]:
paper = res["data"][0]
print(paper["title"], "| citations:", paper.get("citationCount"))

Random Forests | citations: 110290


In [26]:
with open("papers.json", "r", encoding="utf-8") as f:
    payload = json.load(f)

papers = payload.get("data", [])
papers_df = pd.DataFrame(papers)

if "paperId" in papers_df.columns:
    papers_df = papers_df.drop_duplicates(subset=["paperId"]).reset_index(drop=True)

next_token = payload.get("token", None)

print("rows:", len(papers_df))
print("cols:", list(papers_df.columns))
print("next_token:", next_token)

papers_df.head(3)
papers_df.to_csv("../../data/papers/papers_raw_1000.csv", index=False)
papers_df.head(3)

rows: 1000
cols: ['paperId', 'url', 'title', 'venue', 'year', 'referenceCount', 'citationCount', 'openAccessPdf', 'fieldsOfStudy', 's2FieldsOfStudy', 'authors', 'abstract']
next_token: PCOA3RZBB2ADADAF2CVSZVMISWZPLQ2VBCRGZHJBIEAI44DXE2PHQL7NKYHD6KZNQH6KMIMQWXD27XUVYHNN7JW2CQGBSMZHQEVSHZRMYKMODHUKKXWWFIXN7MAUTNIVAQ


,paperId,url,title,venue,year,referenceCount,citationCount,openAccessPdf,fieldsOfStudy,s2FieldsOfStudy,authors,abstract
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,{'url': 'https://link.springer.com/content/pdf...,"[Mathematics, Computer Science]","[{'category': 'Mathematics', 'source': 'extern...","[{'authorId': '2423230', 'name': 'L. Breiman'}]",None
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"{'url': '', 'status': None, 'license': None, '...",[Computer Science],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"{'url': '', 'status': None, 'license': None, '...",[Computer Science],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...


### API를 통해 Paper A -{REF_BY}→ Paper B를 위해 reference 논문 그래프와 정보를 뽑아옴 (각 5개)

- 필요 칼럼: `prereq_paper_id`, `paper_id`, `title`, `year`, `url`, `abstract`, `publication`, `referenceCount`, `citationCount`, `fieldsOfStudy`, `s2FieldsOfStudy`, `intents`, `isInfluential`, `contexts`
    - 만약 papers에서 다운이 안 되었다면 여기에서 가져와서 노드로 등록해야 하기 때문

- 해당 논문을 참고한 citation 논문: prereq_paper_id = 해당 논문, paper_id = citation 논문
- 해당 논문이 참고한 reference 논문: prereq_paper_id = reference 논문, paper_id = 해당 논문
- https://api.semanticscholar.org/api-docs/#tag/Paper-Data/operation/get_graph_get_paper_references
- https://api.semanticscholar.org/api-docs/#tag/Paper-Data/operation/get_graph_get_paper_citations

In [19]:
GRAPH_FIELDS = "contexts,intents,isInfluential,citedPaper.paperId"

BATCH_FIELDS = "paperId,title,year,url,abstract,venue,referenceCount,citationCount,fieldsOfStudy,s2FieldsOfStudy"
COLS = ["ref_paper_id","seed_paper_id","title","year","url","abstract","publication","referenceCount","citationCount","fieldsOfStudy","s2FieldsOfStudy","intents","isInfluential","contexts"]

MIN_INTERVAL, _last = 1.05, 0.0

REF_FETCH = 100          # references에서 먼저 가져올 후보 수
TOP_K = 5                # 최종 저장할 참고문헌 개수(논문당)
## 일단 가지고 있는 논문 1000개 다 처리하되 ref 논문은 각각 5개씩만
## 한 100개 정도 가져온 다음에 그걸 조회해서 influential인데 상위 인용 수 paper만 가져옴

def throttle():
    global _last
    wait = MIN_INTERVAL - (time.time() - _last)
    if wait > 0:
        time.sleep(wait)
    _last = time.time()

def req(method, url, params=None, payload=None, retries=6, timeout=30):
    """
    성공: (json, None)
    실패: (None, 에러메시지 str)
    """
    headers = {"x-api-key": S2_API_KEY}
    last_err = None

    for i in range(retries):
        throttle()
        try:
            r = requests.request(
                method, url,
                headers=headers, params=params, json=payload,
                timeout=timeout
            )

            if r.status_code == 429:
                ra = r.headers.get("Retry-After")
                time.sleep(float(ra) if ra else 1.5 * (2 ** i) + random.uniform(0, 0.3))
                last_err = f"HTTP 429 Too Many Requests (retry {i+1}/{retries})"
                continue

            try:
                r.raise_for_status()
            except requests.HTTPError as e:
                # 응답 본문에 에러 힌트가 있는 경우가 많아서 조금 붙여줌
                body = (r.text or "").strip()
                body = body[:300] + ("..." if len(body) > 300 else "")
                last_err = f"HTTP {r.status_code}: {e}. Body: {body}"
                raise

            return r.json(), None

        except Exception as e:
            if last_err is None:
                last_err = f"{type(e).__name__}: {e}"
            if i == retries - 1:
                return None, last_err
            time.sleep(1.5 * (2 ** i) + random.uniform(0, 0.3))

    return None, last_err or "Unknown error"

def chunks(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def batch_papers(ids, fields, chunk_size=50):
    """
    실패한 chunk가 있더라도 가능한 건 최대한 수집.
    (results, errors) 반환
    """
    if not ids:
        return [], []

    url = "https://api.semanticscholar.org/graph/v1/paper/batch"
    out, errs = [], []

    for c in chunks(ids, chunk_size):
        j, err = req("POST", url, params={"fields": fields}, payload={"ids": c})
        if j:
            out.extend(j)
        else:
            errs.append(f"batch 실패 (ids {len(c)}개): {err}")

    return out, errs

# --- Load seeds ---
with open("papers.json", "r", encoding="utf-8") as f:
    seed_ids = [p.get("paperId") for p in json.load(f).get("data", []) if p.get("paperId")]

rows = []
errors = []  # 에러 로그 모음

# --- Main ---
for seed_id in tqdm(seed_ids, desc="Fetching references"):
    refs, err = req(
        "GET",
        f"https://api.semanticscholar.org/graph/v1/paper/{seed_id}/references",
        params={"fields": GRAPH_FIELDS, "limit": REF_FETCH},
    )
    if not refs:
        errors.append(f"[references 실패] seed={seed_id} :: {err}")
        continue

    data = refs.get("data") or []

    # unique edges by ref paperId (중복 제거)
    seen = set()
    edges = []
    for e in data:
        pid = (e.get("citedPaper") or {}).get("paperId")
        if not pid or pid in seen:
            continue
        seen.add(pid)
        edges.append({
            "pid": pid,
            "intents": e.get("intents", []),
            "inf": e.get("isInfluential", False),
            "contexts": e.get("contexts", []),
        })

    ref_ids = [e["pid"] for e in edges]
    details, batch_errs = batch_papers(ref_ids, BATCH_FIELDS)
    for be in batch_errs:
        errors.append(f"[batch 경고] seed={seed_id} :: {be}")

    meta = {d["paperId"]: d for d in details if d and d.get("paperId")}

    # score & pick top-k: influential 우선, 그 다음 citationCount desc
    for e in edges:
        e["cit"] = (meta.get(e["pid"], {}) or {}).get("citationCount") or 0
    top_edges = sorted(edges, key=lambda x: (x["inf"], x["cit"]), reverse=True)[:TOP_K]

    # 혹시 batch에서 상세를 못 받아온 ref가 섞여있으면 로그 남기기
    for e in top_edges:
        if e["pid"] not in meta:
            errors.append(f"[상세 누락] seed={seed_id} ref={e['pid']} :: batch 결과에 없음")

    for e in top_edges:
        d = meta.get(e["pid"], {}) or {}
        rows.append({
            "seed_paper_id": seed_id,
            "ref_paper_id": e["pid"],
            "title": d.get("title"),
            "year": d.get("year"),
            "url": d.get("url"),
            "abstract": d.get("abstract"),
            "publication": d.get("venue"),
            "referenceCount": d.get("referenceCount"),
            "citationCount": d.get("citationCount"),
            "fieldsOfStudy": d.get("fieldsOfStudy"),
            "s2FieldsOfStudy": d.get("s2FieldsOfStudy"),
            "intents": e["intents"],
            "isInfluential": e["inf"],
            "contexts": e["contexts"],
        })

df = pd.DataFrame(rows, columns=COLS)

print("Done! Total rows:", len(df))
print("Errors:", len(errors))
if errors:
    # 너무 길면 앞부분만 보이게
    print("\n--- Error samples ---")
    for line in errors[:20]:
        print(line)
    if len(errors) > 20:
        print(f"... and {len(errors) - 20} more")

display(df.head())

Fetching references:   0%|          | 0/1000 [00:00<?, ?it/s]

Fetching references: 100%|██████████| 1000/1000 [35:38<00:00,  2.14s/it]

Done! Total rows: 2612
Errors: 0


,prereq_paper_id,paper_id,title,year,url,abstract,publication,referenceCount,citationCount,fieldsOfStudy,s2FieldsOfStudy,intents,isInfluential,contexts
0,NaN,NaN,Attention is All you Need,2017.0,https://www.semanticscholar.org/paper/204e3073...,The dominant sequence transduction models are ...,Neural Information Processing Systems,41,162388,[Computer Science],"[{'category': 'Computer Science', 'source': 'e...",[methodology],True,"[For example, the largest Transformer explored..."
1,NaN,NaN,Deep Contextualized Word Representations,2018.0,https://www.semanticscholar.org/paper/3febb2be...,We introduce a new type of deep contextualized...,North American Chapter of the Association for ...,64,11977,[Computer Science],"[{'category': 'Computer Science', 'source': 'e...","[methodology, background]",True,[BERTLARGE outperforms the authors’ baseline E...
2,NaN,NaN,"SQuAD: 100,000+ Questions for Machine Comprehe...",2016.0,https://www.semanticscholar.org/paper/05dd7254...,We present the Stanford Question Answering Dat...,Conference on Empirical Methods in Natural Lan...,28,8951,[Computer Science],"[{'category': 'Computer Science', 'source': 'e...","[methodology, background]",True,"[…task-specific architectures, ELMo advances t..."
3,NaN,NaN,A Broad-Coverage Challenge Corpus for Sentence...,2017.0,https://www.semanticscholar.org/paper/5ded2b8c...,This paper introduces the Multi-Genre Natural ...,North American Chapter of the Association for ...,56,4843,[Computer Science],"[{'category': 'Computer Science', 'source': 'e...",[background],True,[MNLI Multi-Genre Natural Language Inference i...
4,NaN,NaN,TriviaQA: A Large Scale Distantly Supervised C...,2017.0,https://www.semanticscholar.org/paper/f010affa...,"We present TriviaQA, a challenging reading com...",Annual Meeting of the Association for Computat...,38,3348,[Computer Science],"[{'category': 'Computer Science', 'source': 'e...",[methodology],True,[We therefore use very modest data augmentatio...


In [21]:
errors_df = pd.DataFrame({"error": errors})
display(errors_df.head())

,error


In [22]:
## ref_papers.csv로 저장
df.to_csv("../../data/papers/ref_papers.csv", index=False)

## 노드 생성

In [26]:
## papers와 ref_papers를 불러와서 노드로 저장
## 이때 ref_papers의 ref_paper_id 중 papers에 없는 게 있을 수 있기 때문에 확인해서 nodes에 추가

## cols: ['paperId', 'url', 'title', 'venue', 'year', 'referenceCount', 'citationCount', 'openAccessPdf', 'fieldsOfStudy', 's2FieldsOfStudy', 'authors', 'abstract']
## alias, proposed_concepts, prerequisite_concepts -> 추가하는데 pd.NA로

papers = pd.read_csv("../../data/papers/papers_raw_1000.csv")
ref_papers = pd.read_csv("../../data/papers/ref_papers.csv")

paper_nodes = papers.copy()
if "venue" in paper_nodes.columns:
    paper_nodes = paper_nodes.rename(columns={"venue": "publication"})
ref_nodes = ref_papers.rename(columns={"ref_paper_id": "paperId"}).copy()

drop_cols = [c for c in ["seed_paper_id"] if c in ref_nodes.columns]
ref_nodes = ref_nodes.drop(columns=drop_cols)
nodes = pd.concat([paper_nodes, ref_nodes], ignore_index=True, sort=False)

base_cols = [
    "paperId", "url", "title", "publication", "year",
    "referenceCount", "citationCount",
    "openAccessPdf", "fieldsOfStudy", "s2FieldsOfStudy",
    "authors", "abstract"
]

for c in base_cols:
    if c not in nodes.columns:
        nodes[c] = pd.NA

for c in ["alias", "proposed_concepts", "prerequisite_concepts"]:
    if c not in nodes.columns:
        nodes[c] = pd.NA


nodes = nodes[base_cols + ["alias", "proposed_concepts", "prerequisite_concepts"]].drop_duplicates(
    subset=["paperId"], keep="first"
).reset_index(drop=True)

print("nodes rows:", len(nodes))
print("nodes cols:", list(nodes.columns))
display(nodes.head(3))

nodes.to_csv("../../data/papers/papers_node_paper_1차.csv", index=False)

nodes rows: 2216
nodes cols: ['paperId', 'url', 'title', 'publication', 'year', 'referenceCount', 'citationCount', 'openAccessPdf', 'fieldsOfStudy', 's2FieldsOfStudy', 'authors', 'abstract', 'alias', 'proposed_concepts', 'prerequisite_concepts']


,paperId,url,title,publication,year,referenceCount,citationCount,openAccessPdf,fieldsOfStudy,s2FieldsOfStudy,authors,abstract,alias,proposed_concepts,prerequisite_concepts
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,{'url': 'https://link.springer.com/content/pdf...,"['Mathematics', 'Computer Science']","[{'category': 'Mathematics', 'source': 'extern...","[{'authorId': '2423230', 'name': 'L. Breiman'}]",NaN,<NA>,<NA>,<NA>
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...,<NA>,<NA>,<NA>
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...,<NA>,<NA>,<NA>


In [27]:
paper_ids = set(papers["paperId"].dropna().astype(str))
ref_ids = set(ref_papers["ref_paper_id"].dropna().astype(str))

missing = sorted(ref_ids - paper_ids)
print("ref 중 papers에 없던 paperId 개수:", len(missing))
print("샘플:", missing[:10])

ref 중 papers에 없던 paperId 개수: 1216
샘플: ['0060745e006c5f14ec326904119dca19c6545e51', '007112213ece771be72cbecfd59f048209facabd', '00a8ab985638539533139ad7a9cf45fd9b3cc3de', '00ada6b9969fcc0894c25f5adc7df9fb946856ed', '00f20179b9087fbf24b6656008a9380c590d9ec9', '014576b866078524286802b1d0e18628520aa886', '01b1c227559c7dee2683deca8ed15717db787813', '01d1cef83260339b336969875c3c7fc728c5fd78', '01ee87fed223ee99953879148db7e963ea593036', '024006d4c2a89f7acacc6e4438d156525b60a98f']


## 생성

In [3]:
MODEL = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"  ## "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

llm = LLM(
    model=MODEL,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=512,
)

def extract_json(text: str):
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    # <think>...</think> 제거
    t = re.sub(r"<think>[\s\S]*?</think>", "", t).strip()

    # 코드블록(JSON) 우선 시도
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", t, flags=re.IGNORECASE)
    if m:
        cand = m.group(1).strip()
        # (a) 그대로
        try:
            return json.loads(cand)
        except Exception:
            # (b) 흔한 누락 보정: 마지막 } 없으면 붙여보기
            c2 = cand
            if c2.startswith("{") and not c2.endswith("}"):
                try:
                    return json.loads(c2 + "}")
                except Exception:
                    pass

    # 텍스트에서 JSON 객체 구간 추출
    start = t.find("{")
    if start == -1:
        return None

    # 마지막 }가 있으면 그 구간으로 파싱, 없으면 끝까지를 후보로
    end = t.rfind("}")
    if end != -1 and end > start:
        cand = t[start:end+1].strip()
        try:
            return json.loads(cand)
        except Exception:
            # 마지막 }가 있었는데도 실패하면, 혹시 뒤에 군더더기/잘림 대비 보정도 시도
            pass
    else:
        cand = t[start:].strip()

    # (a) 그대로
    try:
        return json.loads(cand)
    except Exception:
        pass

    # (b) 마지막 } 누락 보정
    if cand.startswith("{") and not cand.endswith("}"):
        try:
            return json.loads(cand + "}")
        except Exception:
            pass

    # (c) 마지막 ]까진 있는데 }만 없는 형태 보정: ...]  -> ...]}
    if cand.startswith("{") and not cand.endswith("}") and cand.rstrip().endswith("]"):
        try:
            return json.loads(cand.rstrip() + "}")
        except Exception:
            pass

    return None


def build_chat_prompt(system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

INFO 01-27 14:07:46 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-27 14:07:48 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-27 14:07:48 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-27 14:07:48 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-27 14:07:48 [model.py:1545] Using max model len 2048


2026-01-27 14:07:50,508	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-27 14:07:50 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-27 14:07:50 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin or gptq_bitblas.


...

INFO 01-27 14:07:58 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-27 14:07:58 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-27 14:07:58 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-27 14:07:58 [vllm.py:765] Cudagraph is disabled under eager mode
WARNING 01-27 14:08:02 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:10 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torc

(EngineCore_DP0 pid=298177) /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=298177) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=298177)   warnings.warn(


(EngineCore_DP0 pid=298177) INFO 01-27 14:08:14 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.94s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:06,  2.02s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:08<00:06,  3.08s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.88s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.54s/it]
(EngineCore_DP0 pid=298177) 


(EngineCore_DP0 pid=298177) INFO 01-27 14:08:36 [default_loader.py:291] Loading weights took 17.81 seconds
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:37 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 24.971637 seconds
(EngineCore_DP0 pid=298177) WARNING 01-27 14:08:38 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:56 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:56 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:56 [kv_cache_utils.py:1310] Maximum concurrency for 2,048 tokens per request: 61.47x
(EngineCore_DP0 pid=298177) INFO 01-27 14:08:56 [core.py:273] init en

#### 1. 논문을 통해 해당 논문의 핵심 KC를 뽑아야 함
- 필요한 칼럼: `논문에서 제안하는 개념`, `논문을 이해하려면 알아야 하는 개념`, `어디서 언급했는지에 대한 evidence`
- `evidence`는 그냥 키워드 매칭으로 가져와도 될 듯 (할루시네이션을 경계해야 함,,,)

In [6]:
## proposed_concepts: 논문이 새로 제안/도입/정의/모델링한 핵심 개념들
## prerequisite_concepts: 이해를 위해 미리 알아야 하는 개념들

papers_df = pd.read_csv("../../data/tmp/papers_node_paper_1차.csv")


KC_PROMPT = """You are an expert research concept extractor.

Given a paper's title and (optionally) abstract, extract two sets of Key Concepts (KC):

1) proposed_concepts:
- Concepts/ideas/methods explicitly introduced, proposed, or newly defined by the paper.
- This includes new models, algorithms, objectives, datasets/benchmarks (only if introduced), training schemes, or novel problem formulations.
- Be conservative: only include what is clearly supported by the title/abstract.

2) prerequisite_concepts:
- Concepts that a reader should know to understand the paper.
- These are background notions implied by the title/abstract, but should still be clearly relevant.

Return ONLY valid, raw JSON. No extra text. No markdown.

### Output schema (must follow exactly)
{
  "proposed_concepts": ["..."],
  "prerequisite_concepts": ["..."]
}

### Rules
- Each field MUST be a list of UNIQUE strings.
- Each concept should be a short noun phrase (2–6 words preferred).
- Avoid overly generic terms like "AI", "deep learning", "neural network" unless the input is truly that generic.
- Do NOT hallucinate details not present in the title/abstract.
- If the abstract is missing or empty:
  - Use ONLY the title evidence.
  - Prefer fewer items (0–6) and be extra conservative.
- Provide 0–12 items per list. If none, use [].

### Input
Title: "{INPUT_TITLE}"
Abstract: "{INPUT_ABSTRACT}"

Output: """


def add_kc_to_nodes(papers_df: pd.DataFrame) -> pd.DataFrame:
    df = papers_df.copy()

    # 안전하게 컬럼명 가정: title, abstract
    df["title"] = df["title"].fillna("").astype(str)
    df["abstract"] = df["abstract"].fillna("").astype(str)

    chat_inputs = []
    row_keys = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="kc: build prompts"):
        title = row["title"].strip()
        abstract = row["abstract"].strip()

        system_msg = KC_PROMPT.replace("{INPUT_TITLE}", title).replace("{INPUT_ABSTRACT}", abstract)
        user_msg = 'Return ONLY raw JSON with keys "proposed_concepts" and "prerequisite_concepts".'
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))
        row_keys.append(i)

    generations = llm.generate(chat_inputs, sampling)

    proposed_map = {}
    prereq_map = {}
    raw_map = {}
    ok_map = {}

    def _clean_list(xs):
        out, seen = [], set()
        for item in xs:
            if not isinstance(item, str):
                continue
            s = item.strip()
            if not s or len(s) > 80:
                continue
            if s not in seen:
                seen.add(s)
                out.append(s)
        return out

    for idx, gen in tqdm(list(zip(row_keys, generations)), total=len(row_keys), desc="kc: parse outputs"):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        ok = (
            isinstance(parsed, dict)
            and isinstance(parsed.get("proposed_concepts"), list)
            and isinstance(parsed.get("prerequisite_concepts"), list)
        )

        raw_map[idx] = raw
        ok_map[idx] = ok

        if ok:
            proposed_map[idx] = _clean_list(parsed["proposed_concepts"])
            prereq_map[idx] = _clean_list(parsed["prerequisite_concepts"])
        else:
            proposed_map[idx] = []
            prereq_map[idx] = []

    df["proposed_concepts"] = df.index.map(lambda i: proposed_map.get(i, []))
    df["prerequisite_concepts"] = df.index.map(lambda i: prereq_map.get(i, []))

    # 디버깅용 컬럼
    df["kc_raw"] = df.index.map(lambda i: raw_map.get(i, ""))
    df["kc_ok"] = df.index.map(lambda i: ok_map.get(i, False))

    return df

In [ ]:
nodes_df = add_kc_to_nodes(papers_df)

display(nodes_df.head(3))

Processed prompts:   0%|          | 0/2216 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
nodes_df

,paperId,url,title,publication,year,referenceCount,citationCount,openAccessPdf,fieldsOfStudy,s2FieldsOfStudy,authors,abstract,alias,proposed_concepts,prerequisite_concepts,kc_raw,kc_ok
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,{'url': 'https://link.springer.com/content/pdf...,"['Mathematics', 'Computer Science']","[{'category': 'Mathematics', 'source': 'extern...","[{'authorId': '2423230', 'name': 'L. Breiman'}]",,NaN,[Random Forests],"[decision trees, ensemble learning, bagging]","{\n ""proposed_concepts"": [""Random Forests""],\...",True
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...,NaN,"[Bidirectional Encoder Representations, Deep b...","[Transformer architecture, Language representa...","{\n ""proposed_concepts"": [\n ""Bidirectiona...",True
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...,NaN,"[very deep convolutional networks, 3x3 convolu...","[convolutional neural networks, image classifi...","{\n ""proposed_concepts"": [\n ""very deep co...",True
3,ad4fd2c149f220a62441576af92a8a669fe81246,https://www.semanticscholar.org/paper/ad4fd2c1...,Scikit-learn: Machine Learning in Python,Journal of machine learning research,2011.0,17,85166,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '2570016', 'name': 'Fabian Pedre...",Scikit-learn is a Python module integrating a ...,NaN,"[scikit-learn package, medium-scale machine le...","[machine learning, supervised learning, unsupe...","{\n ""proposed_concepts"": [\n ""scikit-learn...",True
4,4f8d648c52edf74e41b0996128aa536e13cc7e82,https://www.semanticscholar.org/paper/4f8d648c...,Deep Learning,International Journal of Semantic Computing,2016.0,2,70418,"{'url': '', 'status': 'CLOSED', 'license': Non...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '2343609447', 'name': 'Xingbang ...",,NaN,[],[],"{\n ""proposed_concepts"": [],\n ""prerequisite...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2211,e66496ae422cfd135e66a78e425c0b3fe8d4be22,https://www.semanticscholar.org/paper/e66496ae...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2018.0,36,3752,NaN,"['Psychology', 'Medicine']","[{'category': 'Psychology', 'source': 'externa...",NaN,Problem/Condition Autism spectrum disorder (AS...,NaN,"[ASD prevalence estimates, DSM-IV-TR case defi...","[autism spectrum disorder, DSM-IV-TR criteria,...","{\n ""proposed_concepts"": [\n ""ASD prevalen...",True
2212,ddeb34dc450a7cdc9fd11cdb896b266be9fa1f05,https://www.semanticscholar.org/paper/ddeb34dc...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2020.0,41,2915,NaN,['Medicine'],"[{'category': 'Medicine', 'source': 'external'...",NaN,Problem/Condition Autism spectrum disorder (AS...,NaN,"[ASD prevalence estimation, ADDM Network surve...","[Autism spectrum disorder, Diagnostic and Stat...","{\n ""proposed_concepts"": [\n ""ASD prevalen...",True
2213,d7ac65d335b5d847f4f5826313a8732bc7abc7a8,https://www.semanticsch

In [ ]:
print(nodes_df["kc_ok"].value_counts(dropna=False))
nodes_df.loc[~nodes_df["kc_ok"], ["title", "proposed_concepts", "prerequisite_concepts", "kc_raw"]].head(10)

kc_ok
True    2216
Name: count, dtype: int64


,title,proposed_concepts,prerequisite_concepts,kc_raw


### 1.1 alias 생성

In [ ]:
## kc 펼치기

## 형태: (paperId, name)
## 칼럼: name (= title), edge_type, alias, link, category, paperId, title, abstract
## -> edge_type: ABOUT (proposed_concepts), IN (prerequisite_concepts)
## -> link, category는 나중에 생성 (pd,NA)

def kc_to_nodes_df(papers_df: pd.DataFrame) -> pd.DataFrame:
    base = papers_df.copy()
    base["paperId"] = base["paperId"].astype(str)

    a = base[["paperId","title","abstract","proposed_concepts"]].explode("proposed_concepts")
    a = a.rename(columns={"proposed_concepts":"name", "title":"paper_title"})
    a["edge_type"] = "ABOUT"

    b = base[["paperId","title","abstract","prerequisite_concepts"]].explode("prerequisite_concepts")
    b = b.rename(columns={"prerequisite_concepts":"name", "title":"paper_title"})
    b["edge_type"] = "IN"

    out = pd.concat([a, b], ignore_index=True)
    out["name"] = out["name"].fillna("").astype(str).str.strip()
    out = out[out["name"] != ""]

    out["category"] = pd.NA
    out["link"] = pd.NA

    return out[["name","link","category","edge_type","paperId","paper_title","abstract"]]

nodes_df_kc = kc_to_nodes_df(nodes_df)
display(nodes_df_kc.head(3))

,name,link,category,edge_type,paperId,paper_title,abstract
0,Random Forests,<NA>,<NA>,ABOUT,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,Random Forests,
1,Bidirectional Encoder Representations,<NA>,<NA>,ABOUT,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...
2,Deep bidirectional representations,<NA>,<NA>,ABOUT,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...


In [ ]:
len(nodes_df_kc), nodes_df_kc["name"].nunique()


(18856, 14612)

In [ ]:
def norm_basic(s):
    return " ".join(str(s).strip().lower().split())

nodes_df_kc["name_norm"] = nodes_df_kc["name"].map(norm_basic)
nodes_df_kc["name_norm"].nunique()

14017

In [ ]:
nodes_df_kc["name"].value_counts().head(20)


name
deep learning                    160
convolutional neural networks     93
natural language processing       92
machine learning                  83
deep neural networks              80
image classification              64
neural networks                   61
large language models             60
object detection                  54
supervised learning               54
computer vision                   46
neural network training           44
semantic segmentation             37
unsupervised learning             34
feature extraction                32
deep learning models              30
reinforcement learning            30
transfer learning                 27
adversarial examples              25
recurrent neural networks         25
Name: count, dtype: int64

In [ ]:
## alias 생성

ALIAS_PROMPT = """You are an expert entity normalizer.

Your task is to generate alias surface forms for a given canonical concept name.
These aliases are used to map different user-written strings to the SAME concept node
in a concept graph.

Return ONLY valid, raw JSON. No extra text. No markdown.

### Rules:
- "alias" MUST be a list of UNIQUE strings (no duplicates).
- Do NOT include the canonical name itself.
- Do NOT include trivial variants that are the same as the canonical name
  (case-only changes, extra spaces, or punctuation-only changes).
- If you cannot find at least 1 safe alias, output: {"alias": []}
- Provide 1 to 6 aliases max.

### What counts as an alias
Generate only realistic surface-form variants that people actually write:
- Case variants (upper/lower/mixed)
- Spacing variants (with or without spaces)
- Punctuation variants (hyphens, dots, slashes)
- Acronyms OR full names ONLY if they are unambiguous and commonly used
- Very common typos or misspellings (max 2–3)

Do NOT be creative. Be conservative.

### Examples (for pattern understanding only — do NOT copy)

Input: "machine learning"
Output: {"alias": ["ML", "MachineLearning", "machine-learning"]}

Input: "Wi-Fi"
Output: {"alias": ["wifi", "WiFi", "wi fi"]}

Input: "Seq2Seq"
Output: {"alias": ["seq2seq", "Seq2Seq", "sequence-to-sequence", "sequence to sequence"]}

### Task
Canonical name: "{INPUT_name}"
Output: """

def add_alias_column(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()
    names = df["name"].dropna().astype(str).unique().tolist()

    chat_inputs = []
    for name in tqdm(names, desc="alias: build prompts"):
        system_msg = ALIAS_PROMPT.replace("{INPUT_name}", name)
        user_msg = 'Return ONLY raw JSON in the form {"alias": [...]}'
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    alias_map = {}
    raw_map = {}
    ok_map = {}

    for name, gen in tqdm(list(zip(names, generations)), total=len(names), desc="alias: parse outputs"):
        raw = gen.outputs[0].text.strip()
        raw_map[name] = raw

        parsed = extract_json(raw)
        ok = isinstance(parsed, dict) and isinstance(parsed.get("alias"), list)
        ok_map[name] = ok

        cleaned = []
        if ok:
            dedup = set()
            for item in parsed["alias"]:
                if not isinstance(item, str):
                    continue
                s = item.strip()
                if not s:
                    continue
                if s == name:   # canonical name 제외
                    continue
                if s not in dedup:
                    dedup.add(s)
                    cleaned.append(s)

        alias_map[name] = cleaned

    df["alias"] = df["name"].astype(str).map(alias_map)
    df["alias_raw"] = df["name"].astype(str).map(raw_map)
    df["alias_ok"] = df["name"].astype(str).map(ok_map)
    return df

In [ ]:
nodes_df_kc_alias = add_alias_column(nodes_df_kc)

alias: parse outputs: 100%|██████████| 14612/14612 [00:00<00:00, 94422.76it/s]


In [ ]:
print(nodes_df_kc_alias["alias_ok"].value_counts(dropna=False))
nodes_df_kc_alias.loc[~nodes_df_kc_alias["alias_ok"], ["name", "alias", "alias_raw"]].head(10)

alias_ok
True     18851
False        5
Name: count, dtype: int64


,name,alias,alias_raw
1741,Arcade Learning Environment,[],"{""alias"": [""ALE"", ""arcade learning environment..."
5941,SWAG dataset,[],"{""alias"": [""swag dataset"", ""Swag Dataset"", ""sw..."
9080,Arcade Learning Environment,[],"{""alias"": [""ALE"", ""arcade learning environment..."
10794,algorithmic codesign,[],"{""alias"": [""algorithmic codesign"", ""algorithmi..."
19129,Crystal structure of binding pose,[],"{""alias"": [""crystal structure of binding pose""..."


In [ ]:
for i in range(5): 
    print(nodes_df_kc_alias.iloc[i]["alias_raw"])


{"alias": ["random forests", "RandomForests", "random-forests", "RF", "Random Forest", "random forest"]}
{"alias": ["BERT", "bert", "BidirectionalEncoderRepresentations", "bidirectional encoder representations", "bier", "BiER"]}
{"alias": ["deep bidirectional representations", "deep bidirectional", "deep bidirec", "deep bidirectional rep", "deep bidir", "deep bidir representations"]}
{"alias": ["joint conditioning on left and right context", "joint conditioning left and right context", "joint conditioning on left and right", "joint conditioning left and right", "joint conditioning on l and r context", "joint conditioning l and r context"]}
{"alias": ["fine tuning with one additional output layer", "fine-tuning one output layer", "fine tuning one additional output layer", "one output layer fine tuning", "fine tuning with single additional output layer"]}


### 1.2 wiki랑 비교

In [ ]:
wiki = pd.read_csv("../../data/wikipedia/wiki_subtitle.csv")

## wiki 데이터 내 title과 subtitle을 이용해 link 생성 및 매핑
## 만약 name과 alias의 set 중 하나가 subtitle과 매칭이 되면 https://en.wikipedia.org/wiki/{title}#{subtitle}
## -> name = subtitle, 원래의 name은 alias에 추가됨
## -> name과 alias 중복 제거
## 만약 name과 alias의 set 중 하나가 title과 매칭이 되면 https://en.wikipedia.org/wiki/{title}
## -> name = title, 원래의 name은 alias에 추가됨
## -> name과 alias 중복 제거
## 만약 name과 alias의 set 중 하나가 title과 subtitle과 둘 다 매칭이 안 되면 none

def _norm(s: str) -> str:
    return " ".join(str(s).strip().lower().split())

def _u_path(s: str) -> str:
    # 위키 URL path는 공백을 _로 바꾸는 게 일반적
    return quote(str(s).replace(" ", "_"), safe="()_-.,:")

def _u_frag(s: str) -> str:
    # 섹션 앵커도 보통 공백을 _로 바꿔서 씀
    return quote(str(s).replace(" ", "_"), safe="()_-.,:")

def _parse_listlike(x):
    """ "['a','b']" / "a" / NaN -> list[str] """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v) for v in x]
    s = str(x).strip()
    if not s:
        return []
    # 리스트처럼 생긴 문자열이면 파싱
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [str(i) for i in v]
        except Exception:
            pass
    # 그 외엔 단일 값
    return [s]

def wiki_map(node_df: pd.DataFrame, wiki: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()

    w = wiki.copy()
    w = w.dropna(subset=["title"])
    w["title_raw"] = w["title"].astype(str)
    w["title_n"] = w["title_raw"].map(_norm)

    # 핵심 수정: subtitles를 리스트로 파싱 후 explode ---
    if "subtitles" in w.columns:
        w["subtitle_list"] = w["subtitles"].apply(_parse_listlike)
        w_sub = w.explode("subtitle_list")
        w_sub["subtitle_raw"] = w_sub["subtitle_list"].fillna("").astype(str).str.strip()
        w_sub["subtitle_n"] = w_sub["subtitle_raw"].map(_norm)
        w_sub = w_sub[w_sub["subtitle_n"] != ""]
    else:
        w_sub = w.iloc[0:0].copy()
        w_sub["subtitle_raw"] = ""
        w_sub["subtitle_n"] = ""

    # subtitle -> (title, subtitle) 매핑
    sub_map = {}
    for _, r in w_sub.iterrows():
        # 동일 subtitle이 여러 문서에 있을 수 있는데, 우선 최초 1개만 채택
        sub_map.setdefault(r["subtitle_n"], (r["title_raw"], r["subtitle_raw"]))

    # title -> title 매핑
    title_map = {}
    for _, r in w.iterrows():
        title_map.setdefault(r["title_n"], r["title_raw"])

    def dedup(xs):
        out, seen = [], set()
        for x in xs:
            x = "" if x is None else str(x).strip()
            if not x or x in seen:
                continue
            seen.add(x)
            out.append(x)
        return out

    def _as_list_alias(alias):
        # alias가 문자열로 들어왔는데 리스트처럼 생기면 파싱까지 해줌
        if isinstance(alias, list):
            return alias
        if pd.isna(alias):
            return []
        return _parse_listlike(alias)

    def map_one(my_title, alias):
        alias = _as_list_alias(alias)
        cand = dedup([my_title] + alias)

        # 1) subtitle 매칭
        for x in cand:
            xn = _norm(x)
            if xn in sub_map:
                wiki_title, wiki_sub = sub_map[xn]
                new_title = wiki_sub                      # name을 subtitle로 교체
                new_alias = dedup([a for a in cand if _norm(a) != xn] + [my_title])
                link = f"https://en.wikipedia.org/wiki/{_u_path(wiki_title)}#{_u_frag(wiki_sub)}"
                return new_title, new_alias, link, "subtitle"

        # 2) title 매칭
        for x in cand:
            xn = _norm(x)
            if xn in title_map:
                wiki_title = title_map[xn]
                new_title = wiki_title
                new_alias = dedup([a for a in cand if _norm(a) != xn] + [my_title])
                link = f"https://en.wikipedia.org/wiki/{_u_path(wiki_title)}"
                return new_title, new_alias, link, "title"

        # 3) 미매칭
        return my_title, dedup(alias), None, None

    mapped = df.apply(lambda r: map_one(r.get("name"), r.get("alias", [])),
                      axis=1, result_type="expand")
    mapped.columns = ["name", "alias", "link", "wiki_match_type"]
    df[["name", "alias", "link", "wiki_match_type"]] = mapped
    return df

nodes_df_wiki = wiki_map(nodes_df_kc_alias, wiki)
display(nodes_df_wiki.head(3))

,name,link,category,edge_type,paperId,paper_title,abstract,name_norm,alias,alias_raw,alias_ok,wiki_match_type
0,Random forest,https://en.wikipedia.org/wiki/Random_forest,<NA>,ABOUT,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,Random Forests,,random forests,"[Random Forests, random forests, RandomForests...","{""alias"": [""random forests"", ""RandomForests"", ...",True,title
1,Bidirectional Encoder Representations,None,<NA>,ABOUT,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,bidirectional encoder representations,"[BERT, bert, BidirectionalEncoderRepresentatio...","{""alias"": [""BERT"", ""bert"", ""BidirectionalEncod...",True,None
2,Deep bidirectional representations,None,<NA>,ABOUT,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,deep bidirectional representations,"[deep bidirectional representations, deep bidi...","{""alias"": [""deep bidirectional representations...",True,None


In [ ]:
nodes_df_wiki.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18856 entries, 0 to 19522
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             18856 non-null  object
 1   link             2402 non-null   object
 2   category         0 non-null      object
 3   edge_type        18856 non-null  object
 4   paperId          18856 non-null  object
 5   paper_title      18856 non-null  object
 6   abstract         18856 non-null  object
 7   name_norm        18856 non-null  object
 8   alias            18856 non-null  object
 9   alias_raw        18856 non-null  object
 10  alias_ok         18856 non-null  bool  
 11  wiki_match_type  2402 non-null   object
dtypes: bool(1), object(11)
memory usage: 1.7+ MB


In [ ]:
nodes_df_wiki['link'].nunique()

374

In [ ]:
nodes_df_wiki[nodes_df_wiki['wiki_match_type'] == 'subtitle'].head(5)

,name,link,category,edge_type,paperId,paper_title,abstract,name_norm,alias,alias_raw,alias_ok,wiki_match_type


### 노드 저장

In [ ]:
## 결국 KC만 저장하면 되기 때문에
## name 중복없이 unique하게
## 칼럼: "name","link","category"

def merge_alias(series):
    out, seen = [], set()
    for xs in series:
        if not isinstance(xs, list):
            continue
        for x in xs:
            if isinstance(x, str):
                s = x.strip()
                if s and s not in seen:
                    seen.add(s)
                    out.append(s)
    return out

def pick_link(links):
    links = [x for x in links if isinstance(x, str) and x.strip()]
    if not links:
        return None
    links.sort(key=lambda u: ("#" in u, len(u)), reverse=True)
    return links[0]

nodes_df_kc_all = (
    nodes_df_wiki
    .groupby("name", as_index=False)
    .agg(
        link=("link", pick_link),
        category=("category", "first"),
        alias=("alias", merge_alias),
    )
)

nodes_df_kc_all["alias"] = nodes_df_kc_all.apply(
    lambda r: [a for a in r["alias"] if a != r["name"]],
    axis=1
)

nodes_df_kc_all = nodes_df_kc_all[["name","link","category", "alias"]]
nodes_df_kc_all.to_csv("../../data/papers/papers_node_kc_2차.csv", index=False)

In [ ]:
nodes_df_kc_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14182 entries, 0 to 14181
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      14182 non-null  object
 1   link      374 non-null    object
 2   category  0 non-null      object
dtypes: object(3)
memory usage: 332.5+ KB


In [53]:
nodes_df_kc_all

,name,link,category
0,1 million captioned photographs,None,None
1,1-D convolution,None,None
2,1-nucleotide resolution variant detection,None,None
3,10 land use classes,None,None
4,10 tasks,None,None
...,...,...,...
14177,zero-shot transfer with in-context learning,None,None
14178,zero-shot transfer with locked-image text tuning,None,None
14179,zeroth order optimization,None,None
14180,Γ-point symmetry identification,None,None


In [55]:
df_link_none = nodes_df_kc_all[nodes_df_kc_all["link"].isna()].copy()
df_link_not_none = nodes_df_kc_all[nodes_df_kc_all["link"].notna()].copy()
print("link is None/NaN:", len(df_link_none))
print("link is NOT None/NaN:", len(df_link_not_none))
df_link_not_none

link is None/NaN: 13808
link is NOT None/NaN: 374


,name,link,category
119,3D reconstruction,https://en.wikipedia.org/wiki/3D%20reconstruction,None
172,80 Million Tiny Images,https://en.wikipedia.org/wiki/80%20Million%20T...,None
188,AI alignment,https://en.wikipedia.org/wiki/AI%20alignment,None
208,AI safety,https://en.wikipedia.org/wiki/AI%20safety,None
219,AI-complete,https://en.wikipedia.org/wiki/AI-complete,None
...,...,...,...
4034,WordNet,https://en.wikipedia.org/wiki/WordNet,None
4047,XGBoost,https://en.wikipedia.org/wiki/XGBoost,None
4049,XLNet,https://en.wikipedia.org/wiki/XLNet,None
4074,You Only Look Once,https://en.wikipedia.org/wiki/You%20Only%20Loo...,None


## 엣지 생성

In [56]:
## proposed_concepts(ABOUT): 논문이 새로 제안/도입/정의/모델링한 핵심 개념들
## prerequisite_concepts (IN): 이해를 위해 미리 알아야 하는 개념들

## (paperId, concept) 엣지 DF 만들어야 함
## edges_about 칼럼: paperId, title, abstract, concept (kc_proposed)
## edges_in 칼럼: paperId, title, abstract, concept (kc_prerequisite)

def make_edges_from_nodes(nodes_df_wiki: pd.DataFrame):
    base = nodes_df_wiki.copy()
    base["paperId"] = base["paperId"].astype(str)
    base["concept"] = base["name"].astype(str).str.strip()
    base["title"] = base["paper_title"].fillna("").astype(str)
    base["abstract"] = base["abstract"].fillna("").astype(str)
    base = base[base["concept"] != ""]

    cols = ["paperId", "title", "abstract", "concept"]

    edges_about = (
        base[base["edge_type"] == "ABOUT"][cols]
        .drop_duplicates(subset=["paperId", "concept"])
        .reset_index(drop=True)
    )

    edges_in = (
        base[base["edge_type"] == "IN"][cols]
        .drop_duplicates(subset=["paperId", "concept"])
        .reset_index(drop=True)
    )

    return edges_about, edges_in

edges_about, edges_in = make_edges_from_nodes(nodes_df_wiki)

In [57]:
edges_about.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8425 entries, 0 to 8424
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paperId   8425 non-null   object
 1   title     8425 non-null   object
 2   abstract  8425 non-null   object
 3   concept   8425 non-null   object
dtypes: object(4)
memory usage: 263.4+ KB


In [58]:
edges_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10425 entries, 0 to 10424
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paperId   10425 non-null  object
 1   title     10425 non-null  object
 2   abstract  10425 non-null  object
 3   concept   10425 non-null  object
dtypes: object(4)
memory usage: 325.9+ KB


#### 2.  Paper -{ABOUT}→ KC, KC -{IN}→ Paper
- 필요한 칼럼: `reason`, `strength`

In [59]:
## ABOUT (paper → proposed_concept) 는 “이 논문이 이 개념을 실제로 제안/도입했는가?”
# IN (paper → prerequisite_concept) 는 “이 논문을 이해하려면 이 개념이 선행지식인가?”

EDGE_ABOUT_PROMPT = """You are an expert scientific paper analyst.

Given a PAPER (title + abstract) and a CONCEPT, score whether the paper INTRODUCES/PROPOSES/DEFINES the concept as a main contribution.

Return ONLY valid, raw JSON. No extra text. No markdown.

The output JSON must have exactly these fields:
{
  "strength": number (float between 0 and 1),
  "reason": string (1–2 short sentences, generic, no names, no dates)
}

strength guidelines:
- 0.9–1.0: explicitly introduced/proposed/defined as a key contribution
- 0.6–0.8: strongly central but not explicitly claimed as new
- 0.3–0.5: mentioned as part of method, not a main contribution
- 0.0–0.2: not introduced by the paper

Rules:
- Use ONLY the provided title/abstract; do not assume extra details.
- Be conservative: if unclear, lower the score.

PAPER_TITLE: "{PAPER_TITLE}"
PAPER_ABSTRACT: "{PAPER_ABSTRACT}"
CONCEPT: "{CONCEPT}"

Output:
"""

EDGE_IN_PROMPT = """You are an expert curriculum designer.

Given a PAPER (title + abstract) and a CONCEPT, score how necessary the concept is as prerequisite knowledge to understand the paper.

Return ONLY valid, raw JSON. No extra text. No markdown.

The output JSON must have exactly these fields:
{
  "strength": number (float between 0 and 1),
  "reason": string (1–2 short sentences, generic, no names, no dates)
}

strength guidelines:
- 0.9–1.0: required prerequisite; paper relies heavily on it
- 0.6–0.8: very helpful background
- 0.3–0.5: weak/optional background
- 0.0–0.2: not really a prerequisite

Rules:
- Use ONLY the provided title/abstract; do not assume extra details.
- Be conservative: if unclear, lower the score.

PAPER_TITLE: "{PAPER_TITLE}"
PAPER_ABSTRACT: "{PAPER_ABSTRACT}"
CONCEPT: "{CONCEPT}"

Output:
"""

In [60]:
def add_edge_reason_strength(edge_df: pd.DataFrame, prompt_template: str) -> pd.DataFrame:
    df = edge_df.copy()

    # 유니크 키: (paperId, concept)
    key_df = df[["paperId", "concept"]].dropna().drop_duplicates().astype(str)
    key_pairs = list(key_df.itertuples(index=False, name=None))

    # paperId -> (title, abstract)
    paper_ctx = (
        df[["paperId", "title", "abstract"]]
        .dropna(subset=["paperId"])
        .drop_duplicates(subset=["paperId"])
        .set_index("paperId")[["title", "abstract"]]
        .to_dict(orient="index")
    )

    chat_inputs = []
    for paperId, concept in tqdm(key_pairs, desc="edge: build prompts"):
        ctx = paper_ctx.get(paperId, {"title": "", "abstract": ""})
        system_msg = (
            prompt_template
            .replace("{PAPER_TITLE}", str(ctx.get("title", "")))
            .replace("{PAPER_ABSTRACT}", str(ctx.get("abstract", "")))
            .replace("{CONCEPT}", str(concept))
        )
        user_msg = "Return raw JSON only."
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    # (paperId, concept) -> (strength, reason, raw, ok)
    score_map: dict[tuple[str, str], tuple[object, object, str, bool]] = {}

    for (paperId, concept), gen in tqdm(
        list(zip(key_pairs, generations)),
        total=len(key_pairs),
        desc="edge: parse outputs",
    ):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        ok = isinstance(parsed, dict) and ("strength" in parsed or "reason" in parsed)

        strength = pd.NA
        reason = pd.NA

        if isinstance(parsed, dict):
            s = parsed.get("strength")
            r = parsed.get("reason")

            if isinstance(s, (int, float)):
                strength = float(max(0.0, min(1.0, s)))
            if isinstance(r, str) and r.strip():
                reason = r.strip()

        score_map[(paperId, concept)] = (strength, reason, raw, ok)

    def _get(row, idx):
        key = (str(row["paperId"]), str(row["concept"]))
        return score_map.get(key, (pd.NA, pd.NA, "", False))[idx]

    df["strength"] = df.apply(lambda row: _get(row, 0), axis=1)
    df["reason"]   = df.apply(lambda row: _get(row, 1), axis=1)
    df["edge_raw"] = df.apply(lambda row: _get(row, 2), axis=1)
    df["edge_ok"]  = df.apply(lambda row: _get(row, 3), axis=1)

    return df

In [61]:
edges_about_scored = add_edge_reason_strength(edges_about, EDGE_ABOUT_PROMPT)
edges_about_scored["edge_type"] = "ABOUT"

edges_in_scored = add_edge_reason_strength(edges_in, EDGE_IN_PROMPT)
edges_in_scored["edge_type"] = "IN"

edges_all = pd.concat([edges_about_scored, edges_in_scored], ignore_index=True)

edge: build prompts:   0%|          | 0/8425 [00:00<?, ?it/s]

edge: parse outputs: 100%|██████████| 10425/10425 [00:00<00:00, 110734.78it/s]


In [62]:
edges_all

,paperId,title,abstract,concept,strength,reason,edge_raw,edge_ok,edge_type
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,Random Forests,,Random forest,0.9,The paper introduces random forests as a novel...,"{\n ""strength"": 0.9,\n ""reason"": ""The paper ...",True,ABOUT
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,Bidirectional Encoder Representations,0.95,The paper explicitly introduces BERT as Bidire...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,ABOUT
2,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,Deep bidirectional representations,0.95,The paper explicitly introduces BERT as a mode...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,ABOUT
3,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,Joint conditioning on left and right context,0.95,The paper explicitly introduces joint conditio...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,ABOUT
4,df2b0e26d0599ce3e70df8a9da02e51594e0e992,BERT: Pre-training of Deep Bidirectional Trans...,We introduce a new language representation mod...,Fine-tuning with one additional output layer,0.9,The paper explicitly introduces fine-tuning wi...,"{\n ""strength"": 0.9,\n ""reason"": ""The paper ...",True,ABOUT
...,...,...,...,...,...,...,...,...,...
18845,47df3fd32d00220c85c2c51a571254fd99b2ecc7,MetaICL: Learning to Learn In Context,We introduce MetaICL (Meta-training for In-Con...,few-shot learning,0.95,"The paper centers on few-shot learning, with M...","{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,IN
18846,47df3fd32d00220c85c2c51a571254fd99b2ecc7,MetaICL: Learning to Learn In Context,We introduce MetaICL (Meta-training for In-Con...,pretrained language model,0.95,The paper centers on training a pretrained lan...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,IN
18847,47df3fd32d00220c85c2c51a571254fd99b2ecc7,MetaICL: Learning to Learn In Context,We introduce MetaICL (Meta-training for In-Con...,zero-shot transfer,0.8,Zero-shot transfer is a key baseline in the pa...,"{\n ""strength"": 0.8,\n ""reason"": ""Zero-shot ...",True,IN
18848,47df3fd32d00220c85c2c51a571254fd99b2ecc7,MetaICL: Learning to Learn In Context,We introduce MetaICL (Meta-training for In-Con...,Natural language processing,0.95,The paper focuses on in-context learning withi...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",True,IN


In [63]:
print(edges_all["edge_ok"].value_counts(dropna=False))
edges_all.loc[~edges_all["edge_ok"]].head(10)

edge_ok
True     18846
False        4
Name: count, dtype: int64


,paperId,title,abstract,concept,strength,reason,edge_raw,edge_ok,edge_type
678,de5e7320729f5d3cbb6709eb6329ec41ace8c95d,Gaussian Error Linear Units (GELUs),We propose the Gaussian Error Linear Unit (GEL...,Gaussian cumulative distribution weighting,<NA>,<NA>,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",False,ABOUT
17477,8733fe2371b615609b04e2e910b1ecfa8e77cbc2,Square Attack: a query-efficient black-box adv...,"We propose the Square Attack, a score-based bl...",l_2 adversarial perturbation,<NA>,<NA>,"{\n ""strength"": 0.9,\n ""reason"": ""The paper ...",False,IN
17478,8733fe2371b615609b04e2e910b1ecfa8e77cbc2,Square Attack: a query-efficient black-box adv...,"We propose the Square Attack, a score-based bl...",l_infinity adversarial perturbation,<NA>,<NA>,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",False,IN
17487,b3f1aa12dde233aaf543bb9ccb27213c494e0fd5,Unlabeled Data Improves Adversarial Robustness,"We demonstrate, theoretically and empirically,...",l_infinity robustness,<NA>,<NA>,"{\n ""strength"": 0.95,\n ""reason"": ""The paper...",False,IN


In [64]:
## 1차 저장

edges_all[["paperId", "concept", "edge_type", "strength", "reason", "edge_raw"]].to_csv("../../data/papers/papers_edge_paper_kc_1차.csv", index=False)

#### 3. Paper A -{REF_BY}→ Paper B
- 필요한 칼럼: `strength`

In [74]:
## max_model_len 2048이 넘어서


MODEL = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"  ## "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

llm = LLM(
    model=MODEL,
    trust_remote_code=True,
    max_model_len=3200,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=512,
)

def extract_json(text: str):
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    # <think>...</think> 제거
    t = re.sub(r"<think>[\s\S]*?</think>", "", t).strip()

    # 코드블록(JSON) 우선 시도
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", t, flags=re.IGNORECASE)
    if m:
        cand = m.group(1).strip()
        # (a) 그대로
        try:
            return json.loads(cand)
        except Exception:
            # (b) 흔한 누락 보정: 마지막 } 없으면 붙여보기
            c2 = cand
            if c2.startswith("{") and not c2.endswith("}"):
                try:
                    return json.loads(c2 + "}")
                except Exception:
                    pass

    # 텍스트에서 JSON 객체 구간 추출
    start = t.find("{")
    if start == -1:
        return None

    # 마지막 }가 있으면 그 구간으로 파싱, 없으면 끝까지를 후보로
    end = t.rfind("}")
    if end != -1 and end > start:
        cand = t[start:end+1].strip()
        try:
            return json.loads(cand)
        except Exception:
            # 마지막 }가 있었는데도 실패하면, 혹시 뒤에 군더더기/잘림 대비 보정도 시도
            pass
    else:
        cand = t[start:].strip()

    # (a) 그대로
    try:
        return json.loads(cand)
    except Exception:
        pass

    # (b) 마지막 } 누락 보정
    if cand.startswith("{") and not cand.endswith("}"):
        try:
            return json.loads(cand + "}")
        except Exception:
            pass

    # (c) 마지막 ]까진 있는데 }만 없는 형태 보정: ...]  -> ...]}
    if cand.startswith("{") and not cand.endswith("}") and cand.rstrip().endswith("]"):
        try:
            return json.loads(cand.rstrip() + "}")
        except Exception:
            pass

    return None


def build_chat_prompt(system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

INFO 01-26 17:52:45 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 3200, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-26 17:52:46 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-26 17:52:46 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-26 17:52:46 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-26 17:52:46 [model.py:1545] Using max model len 3200
INFO 01-26 17:52:46 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files: 100%|██████████| 5/5 [00:06<00:00,  1.29s/it]


WARNING 01-26 17:52:55 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-26 17:52:55 [vllm.py:765] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:03 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=3200, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=gptq, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_pars

(EngineCore_DP0 pid=237885) /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=237885) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=237885)   warnings.warn(


(EngineCore_DP0 pid=237885) INFO 01-26 17:53:08 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.97s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:06,  2.05s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:09<00:06,  3.13s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  3.96s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  3.61s/it]
(EngineCore_DP0 pid=237885) 


(EngineCore_DP0 pid=237885) INFO 01-26 17:53:29 [default_loader.py:291] Loading weights took 18.15 seconds
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:30 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 24.624884 seconds
(EngineCore_DP0 pid=237885) WARNING 01-26 17:53:31 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:48 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:49 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:49 [kv_cache_utils.py:1310] Maximum concurrency for 3,200 tokens per request: 39.34x
(EngineCore_DP0 pid=237885) INFO 01-26 17:53:49 [core.py:273] init en

In [75]:
EDGE_REF_BY_PROMPT = """You are an expert scientific paper analyst.

Given a TARGET paper (the one we want to understand) and a REFERENCED paper,
score how essential the referenced paper is for understanding the target paper.

Return ONLY valid, raw JSON. No extra text. No markdown.
Output schema:
{"strength": number}

strength guidelines (0~1):
- 0.9–1.0: essential; heavily relied upon
- 0.6–0.8: very helpful; key background/method
- 0.3–0.5: somewhat relevant; minor support
- 0.0–0.2: weak/optional

Use ONLY the provided text. Be conservative if unclear.

TARGET_TITLE: "{TARGET_TITLE}"
TARGET_ABSTRACT: "{TARGET_ABSTRACT}"

REF_TITLE: "{REF_TITLE}"
REF_ABSTRACT: "{REF_ABSTRACT}"

CITATION_INTENTS: {INTENTS}
IS_INFLUENTIAL: {IS_INF}
CONTEXTS: {CONTEXTS}

Output: """

def add_ref_by_strength(ref_df: pd.DataFrame, seeds_json_path="papers.json") -> pd.DataFrame:
    df = ref_df.copy()

    with open(seeds_json_path, "r", encoding="utf-8") as f:
        seeds = json.load(f).get("data", [])

    seed_ctx = {
        s["paperId"]: {
            "title": (s.get("title") or ""),
            "abstract": (s.get("abstract") or "")
        }
        for s in seeds if s.get("paperId")
    }

    key_df = df[["seed_paper_id", "ref_paper_id"]].dropna().drop_duplicates().astype(str)
    key_pairs = list(key_df.itertuples(index=False, name=None))

    ref_ctx = (
        df[["ref_paper_id", "title", "abstract"]]
        .drop_duplicates(subset=["ref_paper_id"])
        .set_index("ref_paper_id")[["title", "abstract"]]
        .to_dict(orient="index")
    )

    df0 = df.copy()
    df0["seed_paper_id"] = df0["seed_paper_id"].astype(str)
    df0["ref_paper_id"] = df0["ref_paper_id"].astype(str)

    chat_inputs = []
    for seed_id, ref_id in tqdm(key_pairs, desc="ref_by: build prompts"):
        t = seed_ctx.get(seed_id, {"title": "", "abstract": ""})
        r = ref_ctx.get(ref_id, {"title": "", "abstract": ""})

        row0 = df0[(df0["seed_paper_id"] == seed_id) & (df0["ref_paper_id"] == ref_id)].iloc[0]

        system_msg = (
            EDGE_REF_BY_PROMPT
            .replace("{TARGET_TITLE}", str(t.get("title", "")))
            .replace("{TARGET_ABSTRACT}", str(t.get("abstract", "")))
            .replace("{REF_TITLE}", str(r.get("title", "")))
            .replace("{REF_ABSTRACT}", str(r.get("abstract", "")))
            .replace("{INTENTS}", str(row0.get("intents", [])))
            .replace("{IS_INF}", str(bool(row0.get("isInfluential", False))))
            .replace("{CONTEXTS}", str(row0.get("contexts", [])[:3]))
        )
        chat_inputs.append(build_chat_prompt(system_msg, "Return raw JSON only."))

    generations = llm.generate(chat_inputs, sampling)

    score_map = {}
    for (seed_id, ref_id), gen in tqdm(list(zip(key_pairs, generations)), total=len(key_pairs), desc="ref_by: parse"):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        strength = pd.NA
        if isinstance(parsed, dict) and isinstance(parsed.get("strength"), (int, float)):
            strength = float(max(0.0, min(1.0, parsed["strength"])))

        score_map[(seed_id, ref_id)] = strength

    df["strength"] = df.apply(
        lambda r: score_map.get((str(r["seed_paper_id"]), str(r["ref_paper_id"])), pd.NA),
        axis=1
    )
    return df


In [76]:
ref_df = pd.read_csv("../../data/papers/ref_papers.csv")

df_scored = add_ref_by_strength(ref_df, seeds_json_path="papers.json")

ref_by: parse: 100%|██████████| 2612/2612 [00:00<00:00, 113888.68it/s]


In [77]:
out = df_scored[[
    "seed_paper_id", "ref_paper_id", "intents", "isInfluential", "contexts", "strength"
]].copy()
out["name"] = "REF_BY"

out = out[["name","ref_paper_id","seed_paper_id","intents","isInfluential","contexts","strength"]]
out.to_csv("../../data/papers/papers_edge_ref_by_1차.csv", index=False)

## 정리

### 1) paper node

In [ ]:
## 그냥 paper 가져온 거랑 ref 합쳐서 중복 없이 저장해놓은 노드파일
papers_node_paper = pd.read_csv("../../data/papers/papers_node_paper_1차.csv")
papers_node_paper.head()

,paperId,url,title,publication,year,referenceCount,citationCount,openAccessPdf,fieldsOfStudy,s2FieldsOfStudy,authors,abstract,alias,proposed_concepts,prerequisite_concepts
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,{'url': 'https://link.springer.com/content/pdf...,"['Mathematics', 'Computer Science']","[{'category': 'Mathematics', 'source': 'extern...","[{'authorId': '2423230', 'name': 'L. Breiman'}]",NaN,NaN,NaN,NaN
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...,NaN,NaN,NaN
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...,NaN,NaN,NaN
3,ad4fd2c149f220a62441576af92a8a669fe81246,https://www.semanticscholar.org/paper/ad4fd2c1...,Scikit-learn: Machine Learning in Python,Journal of machine learning research,2011.0,17,85166,"{'url': '', 'status': None, 'license': None, '...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '2570016', 'name': 'Fabian Pedre...",Scikit-learn is a Python module integrating a ...,NaN,NaN,NaN
4,4f8d648c52edf74e41b0996128aa536e13cc7e82,https://www.semanticscholar.org/paper/4f8d648c...,Deep Learning,International Journal of Semantic Computing,2016.0,2,70418,"{'url': '', 'status': 'CLOSED', 'license': Non...",['Computer Science'],"[{'category': 'Computer Science', 'source': 'e...","[{'authorId': '2343609447', 'name': 'Xingbang ...",NaN,NaN,NaN,NaN


In [ ]:
papers_node_paper.columns

papers_node_paper = papers_node_paper[['paperId', 'url', 'title', 'publication', 'year', 'referenceCount',
       'citationCount', 'fieldsOfStudy', 's2FieldsOfStudy',
       'authors', 'abstract']]

def to_list(x):
    if pd.isna(x): return []
    try: return x if isinstance(x, list) else ast.literal_eval(str(x))
    except: return []

papers_node_paper["categories"] = papers_node_paper.apply(
    lambda r: list(dict.fromkeys(
        [v for v in to_list(r["fieldsOfStudy"]) if isinstance(v, str)] +
        [d.get("category") for d in to_list(r["s2FieldsOfStudy"])
         if isinstance(d, dict) and isinstance(d.get("category"), str)]
    )),
    axis=1
)

papers_node_paper = papers_node_paper.drop(columns=["fieldsOfStudy", "s2FieldsOfStudy"])

In [ ]:
papers_node_paper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2216 entries, 0 to 2215
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   paperId         2216 non-null   object 
 1   url             2216 non-null   object 
 2   title           2216 non-null   object 
 3   publication     1946 non-null   object 
 4   year            2200 non-null   float64
 5   referenceCount  2216 non-null   int64  
 6   citationCount   2216 non-null   int64  
 7   authors         1000 non-null   object 
 8   abstract        1224 non-null   object 
 9   categories      2216 non-null   object 
dtypes: float64(1), int64(2), object(7)
memory usage: 173.2+ KB


In [ ]:
papers_node_paper.to_csv("../../data/papers/papers_node_paper.csv", index=False)

### 2) paper에서 얻은 KC

In [ ]:
## KC 생성 및 alias 생성한 후 wiki와 비교해서 표제어로 바뀐 노드파일
papers_node_kc = pd.read_csv("../../data/papers/papers_node_kc_with_category.csv")
papers_node_kc.head()

,name,link,category
0,1 million captioned photographs,NaN,NaN
1,1-D convolution,NaN,NaN
2,1-nucleotide resolution variant detection,NaN,NaN
3,10 land use classes,NaN,NaN
4,10 tasks,NaN,NaN


In [ ]:
papers_node_kc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14182 entries, 0 to 14181
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      14182 non-null  object
 1   link      2114 non-null   object
 2   category  2114 non-null   object
dtypes: object(3)
memory usage: 332.5+ KB


In [ ]:
## wiki에서 카테고리 뽑아오기

## 그 후에 저장
papers_node_kc.to_csv("../../data/papers/papers_node_kc.csv", index=False)

### 3) paper and KC edge

In [ ]:
## paper-KC 엣지 
papers_edge_paper_kc = pd.read_csv("../../data/papers/papers_edge_paper_kc_1차.csv")   
papers_edge_paper_kc.head()

,paperId,concept,edge_type,strength,reason,edge_raw
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,Random forest,ABOUT,0.90,The paper introduces random forests as a novel...,"{\n ""strength"": 0.9,\n ""reason"": ""The paper ..."
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,Bidirectional Encoder Representations,ABOUT,0.95,The paper explicitly introduces BERT as Bidire...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper..."
2,df2b0e26d0599ce3e70df8a9da02e51594e0e992,Deep bidirectional representations,ABOUT,0.95,The paper explicitly introduces BERT as a mode...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper..."
3,df2b0e26d0599ce3e70df8a9da02e51594e0e992,Joint conditioning on left and right context,ABOUT,0.95,The paper explicitly introduces joint conditio...,"{\n ""strength"": 0.95,\n ""reason"": ""The paper..."
4,df2b0e26d0599ce3e70df8a9da02e51594e0e992,Fine-tuning with one additional output layer,ABOUT,0.90,The paper explicitly introduces fine-tuning wi...,"{\n ""strength"": 0.9,\n ""reason"": ""The paper ..."


In [ ]:
papers_edge_paper_kc.info()

## 결측치 처리 -> edge_raw를 활용해 손수 채워주기
## strength, reason 각 4개

'''
de5e7320729f5d3cbb6709eb6329ec41ace8c95d,Gaussian cumulative distribution weighting,ABOUT,0.95,"The paper explicitly introduces GELU as a new activation function based on Gaussian cumulative distribution weighting. The core formulation $x\Phi(x)$ directly defines the concept as a key contribution.","{
  ""strength"": 0.95,
  ""reason"": ""The paper explicitly introduces GELU as a new activation function based on Gaussian cumulative distribution weighting. The core formulation $x\Phi(x)$ directly defines the concept as a key contribution.""
}"
8733fe2371b615609b04e2e910b1ecfa8e77cbc2,l_2 adversarial perturbation,IN,0.9,"The paper focuses on $l_2$- and $l_\infty$-adversarial attacks, and the $l_2$ perturbation is central to the attack's design and evaluation. Without understanding $l_2$ perturbations, the core mechanism and results cannot be grasped.","{
  ""strength"": 0.9,
  ""reason"": ""The paper focuses on $l_2$- and $l_\infty$-adversarial attacks, and the $l_2$ perturbation is central to the attack's design and evaluation. Without understanding $l_2$ perturbations, the core mechanism and results cannot be grasped.""
}"
8733fe2371b615609b04e2e910b1ecfa8e77cbc2,l_infinity adversarial perturbation,IN,0.95,"The paper focuses on $l_\infty$-adversarial attacks, and the proposed method is specifically designed for $l_\infty$-bounded perturbations. Understanding $l_\infty$ adversarial perturbations is essential to grasp the attack's formulation and evaluation.","{
  ""strength"": 0.95,
  ""reason"": ""The paper focuses on $l_\infty$-adversarial attacks, and the proposed method is specifically designed for $l_\infty$-bounded perturbations. Understanding $l_\infty$ adversarial perturbations is essential to grasp the attack's formulation and evaluation.""
}"
b3f1aa12dde233aaf543bb9ccb27213c494e0fd5,l_infinity robustness,IN,0.95,"The paper's core claims revolve around achieving improved robustness in the $\ell_\infty$ norm, and the results are explicitly evaluated against $\ell_\infty$ attacks. Without understanding $\ell_\infty$ robustness, the significance of the results cannot be grasped.","{
  ""strength"": 0.95,
  ""reason"": ""The paper's core claims revolve around achieving improved robustness in the $\ell_\infty$ norm, and the results are explicitly evaluated against $\ell_\infty$ attacks. Without understanding $\ell_\infty$ robustness, the significance of the results cannot be grasped.""
}"

'''

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18850 entries, 0 to 18849
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   paperId    18850 non-null  object 
 1   concept    18850 non-null  object 
 2   edge_type  18850 non-null  object 
 3   strength   18850 non-null  float64
 4   reason     18850 non-null  object 
 5   edge_raw   18850 non-null  object 
dtypes: float64(1), object(5)
memory usage: 883.7+ KB


'\nde5e7320729f5d3cbb6709eb6329ec41ace8c95d,Gaussian cumulative distribution weighting,ABOUT,0.95,"The paper explicitly introduces GELU as a new activation function based on Gaussian cumulative distribution weighting. The core formulation $x\\Phi(x)$ directly defines the concept as a key contribution.","{\n  ""strength"": 0.95,\n  ""reason"": ""The paper explicitly introduces GELU as a new activation function based on Gaussian cumulative distribution weighting. The core formulation $x\\Phi(x)$ directly defines the concept as a key contribution.""\n}"\n8733fe2371b615609b04e2e910b1ecfa8e77cbc2,l_2 adversarial perturbation,IN,0.9,"The paper focuses on $l_2$- and $l_\\infty$-adversarial attacks, and the $l_2$ perturbation is central to the attack\'s design and evaluation. Without understanding $l_2$ perturbations, the core mechanism and results cannot be grasped.","{\n  ""strength"": 0.9,\n  ""reason"": ""The paper focuses on $l_2$- and $l_\\infty$-adversarial attacks, and the $l_2$ pertur

In [ ]:
papers_edge_paper_kc.columns

Index(['paperId', 'concept', 'edge_type', 'strength', 'reason', 'edge_raw'], dtype='object')

In [ ]:
papers_edge_paper_kc = (
    papers_edge_paper_kc
    .rename(columns={'edge_type': 'name'})
    [['paperId', 'concept', 'name', 'strength', 'reason']]
)

papers_edge_paper_kc.to_csv("../../data/papers/papers_edge_paper_kc.csv", index=False)

### 4) paper and paper edge

In [ ]:
## paper-paper 엣지
papers_edge_paper_paper = pd.read_csv("../../data/papers/papers_edge_ref_by_1차.csv")
papers_edge_paper_paper.head()

,name,ref_paper_id,seed_paper_id,intents,isInfluential,contexts,strength
0,REF_BY,204e3073870fae3d05bcbc2f6a8e263d9b72e776,df2b0e26d0599ce3e70df8a9da02e51594e0e992,['methodology'],True,"['For example, the largest Transformer explore...",0.9
1,REF_BY,3febb2bed8865945e7fddc99efd791887bb7e14f,df2b0e26d0599ce3e70df8a9da02e51594e0e992,"['methodology', 'background']",True,['BERTLARGE outperforms the authors’ baseline ...,0.8
2,REF_BY,05dd7254b632376973f3a1b4d39485da17814df5,df2b0e26d0599ce3e70df8a9da02e51594e0e992,"['methodology', 'background']",True,"['…task-specific architectures, ELMo advances ...",0.8
3,REF_BY,5ded2b8c64491b4a67f6d39ce473d4b9347a672e,df2b0e26d0599ce3e70df8a9da02e51594e0e992,['background'],True,['MNLI Multi-Genre Natural Language Inference ...,0.8
4,REF_BY,f010affab57b5fcf1cd6be23df79d8ec98c7289c,df2b0e26d0599ce3e70df8a9da02e51594e0e992,['methodology'],True,['We therefore use very modest data augmentati...,0.3


In [ ]:
papers_edge_paper_paper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2612 entries, 0 to 2611
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           2612 non-null   object 
 1   ref_paper_id   2612 non-null   object 
 2   seed_paper_id  2612 non-null   object 
 3   intents        2612 non-null   object 
 4   isInfluential  2612 non-null   bool   
 5   contexts       2612 non-null   object 
 6   strength       2612 non-null   float64
dtypes: bool(1), float64(1), object(5)
memory usage: 125.1+ KB


In [ ]:
papers_edge_paper_paper.to_csv("../../data/papers/papers_edge_paper_paper.csv", index=False)